# CLAP Explainability with SHAP, LIME, and Captum

This notebook demonstrates how to apply explainability techniques to the **CLAP** (Contrastive Language-Audio Pretraining) model from LAION.

CLAP is the audio equivalent of CLIP — it maps audio and text into a shared embedding space using:
- A **SWINTransformer** to encode log-Mel spectrograms
- A **RoBERTa** model to encode text

Similarity between audio and text is measured via dot product in this shared latent space.

### Why explainability is non-trivial for CLAP

`get_audio_features()` returns a raw **512-dimensional embedding** — not a probability or score. SHAP and LIME both need a *scalar output* to explain. So the key trick throughout this notebook is to **define a downstream task** (zero-shot classification via cosine similarity) that gives us a score to attribute.

### Three approaches covered
| Approach | Tool | Granularity |
|---|---|---|
| Time-segment attribution | LIME | Which part of the waveform matters? |
| Time-frequency attribution | SHAP KernelExplainer | Which spectrogram regions matter? |
| Gradient attribution | Captum IntegratedGradients | Fast, faithful to model internals |

## 0. Install dependencies

Run this cell once to install everything needed.

In [1]:
!pip install transformers datasets torch torchaudio librosa \
             shap lime captum matplotlib numpy -q

## 1. Load the CLAP model and a sample audio clip

We load `laion/larger_clap_general` — a CLAP checkpoint trained on general audio, music, and speech.
The sample audio comes from the LibriSpeech validation set (English speech).

In [ ]:
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import ClapModel, ClapProcessor

# Load model and processor
model = ClapModel.from_pretrained("laion/larger_clap_general")
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")
model.eval()

# Load a sample audio clip (LibriSpeech — English speech)
librispeech_dummy = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
)
audio_sample = librispeech_dummy[0]
audio_array = audio_sample["audio"]["array"].astype(np.float32)
sampling_rate = audio_sample["audio"]["sampling_rate"]

print(f"Audio duration : {len(audio_array) / sampling_rate:.2f}s")
print(f"Sampling rate  : {sampling_rate} Hz")
print(f"Samples        : {len(audio_array)}")

# Plot the waveform
plt.figure(figsize=(12, 2))
plt.plot(np.linspace(0, len(audio_array)/sampling_rate, len(audio_array)), audio_array, linewidth=0.5)
plt.title("Input waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

## 2. Define candidate labels and the scoring function

This is the core building block used by all three explainability methods.

**How it works:**
1. Encode the audio → 512-dim embedding
2. Encode each text label → 512-dim embedding
3. Compute cosine similarity between audio and each label
4. Apply softmax to get a probability distribution over labels

This gives us a scalar per label — exactly what SHAP/LIME need.

In [ ]:
# Define zero-shot classification labels
# Adjust these to whatever categories are relevant for your use case
CANDIDATE_LABELS = ["speech", "music", "noise", "dog barking"]

def predict_proba(audio_arrays):
    """
    Score a batch of 1D audio arrays against CANDIDATE_LABELS.
    
    Args:
        audio_arrays: list of 1D numpy arrays (raw waveforms)
    Returns:
        np.array of shape [batch, num_labels] with softmax probabilities
    """
    results = []
    
    # Precompute text embeddings once (they don't change per sample)
    text_inputs = processor(
        text=CANDIDATE_LABELS, return_tensors="pt", padding=True
    )
    with torch.no_grad():
        text_emb = model.get_text_features(**text_inputs)          # [num_labels, 512]
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)  # L2 normalize

    for audio in audio_arrays:
        audio_inputs = processor(
            audios=audio, return_tensors="pt", sampling_rate=sampling_rate
        )
        with torch.no_grad():
            audio_emb = model.get_audio_features(**audio_inputs)       # [1, 512]
            audio_emb = audio_emb / audio_emb.norm(dim=-1, keepdim=True)

        # Cosine similarity scaled by learned logit_scale
        logits = (audio_emb @ text_emb.T) * model.logit_scale.exp()   # [1, num_labels]
        probs  = logits.softmax(dim=-1).squeeze().numpy()              # [num_labels]
        results.append(probs)

    return np.array(results)  # [batch, num_labels]


# Quick sanity check on our original audio
scores = predict_proba([audio_array])[0]
print("Zero-shot classification scores:")
for label, score in zip(CANDIDATE_LABELS, scores):
    bar = "█" * int(score * 40)
    print(f"  {label:<15} {score:.3f}  {bar}")

## 3. Approach 1 — LIME on time segments

**What LIME does:** It creates perturbed versions of the input by randomly masking segments, observes how the model output changes, and fits a simple linear model to approximate local feature importance.

**For audio:** We split the waveform into N time segments and treat each segment as one feature. LIME masks segments (replaces them with silence/zeros) and measures how much each segment contributes to the predicted label.

**Intuition:** A segment with high positive LIME weight means "when this part of the audio is present, the model is more confident in this label."

In [ ]:
from lime import lime_tabular

# ── Configuration ────────────────────────────────────────────
N_SEGMENTS = 50   # Split audio into 50 equal time segments
# ─────────────────────────────────────────────────────────────

segment_len = len(audio_array) // N_SEGMENTS

def audio_to_segments(audio):
    """Summarise each segment as its mean amplitude."""
    return np.array([
        audio[i*segment_len : (i+1)*segment_len].mean()
        for i in range(N_SEGMENTS)
    ])

def reconstruct_from_segments(segment_matrix):
    """
    LIME passes a matrix of perturbed segment vectors.
    We reconstruct a full waveform for each row by tiling segment means,
    then score it.
    """
    audios = []
    for row in segment_matrix:
        reconstructed = np.repeat(row, segment_len).astype(np.float32)
        # Trim/pad to original length
        reconstructed = reconstructed[:len(audio_array)]
        audios.append(reconstructed)
    return predict_proba(audios)

# Build segment representation of our audio
segments = audio_to_segments(audio_array).reshape(1, -1)  # [1, N_SEGMENTS]

# Create LIME explainer
explainer = lime_tabular.LimeTabularExplainer(
    segments,
    feature_names=[f"seg_{i}" for i in range(N_SEGMENTS)],
    class_names=CANDIDATE_LABELS,
    mode="classification",
)

# Explain the top predicted label
top_label_idx = scores.argmax()
print(f"Explaining label: '{CANDIDATE_LABELS[top_label_idx]}'")

exp = explainer.explain_instance(
    segments[0],
    reconstruct_from_segments,
    num_features=N_SEGMENTS,
    labels=[top_label_idx],
    num_samples=200,   # increase for more stable estimates (slower)
)

# Extract and plot LIME weights
lime_weights = dict(exp.as_list(label=top_label_idx))
seg_indices  = [int(k.split("_")[1]) for k in lime_weights]
weights      = [lime_weights[k] for k in lime_weights]

# Map weights back to time axis
weight_array = np.zeros(N_SEGMENTS)
for k, w in zip(seg_indices, weights):
    weight_array[k] = w

time_axis = np.linspace(0, len(audio_array)/sampling_rate, N_SEGMENTS)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)

# Waveform
axes[0].plot(
    np.linspace(0, len(audio_array)/sampling_rate, len(audio_array)),
    audio_array, linewidth=0.5, color="steelblue"
)
axes[0].set_title("Waveform")
axes[0].set_ylabel("Amplitude")

# LIME weights
colors = ["tomato" if w < 0 else "seagreen" for w in weight_array]
axes[1].bar(time_axis, weight_array, width=time_axis[1]-time_axis[0],
            color=colors, align="edge")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_title(f"LIME segment importance for label: '{CANDIDATE_LABELS[top_label_idx]}'")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("LIME weight")

plt.tight_layout()
plt.show()
print("Green = supports this label | Red = opposes this label")

## 4. Approach 2 — SHAP on the log-Mel spectrogram

**What SHAP does:** It uses Shapley values from cooperative game theory to fairly distribute the model's prediction among all input features.

**For audio:** Instead of the raw waveform, we work on the **log-Mel spectrogram** — the actual input representation that CLAP's SWINTransformer sees. Each cell in the spectrogram is a (time, frequency) bin, and SHAP tells us which bins were most influential.

**Intuition:** High SHAP values in a frequency band at a specific time mean "this combination of time + frequency pushed the model toward this label."

> ⚠️ KernelExplainer is slow because it samples many random coalitions. We use a small `nsamples` here for demonstration — increase it for more accurate estimates.

In [ ]:
import shap

# ── Configuration ────────────────────────────────────────────
N_MELS      = 64   # Mel frequency bins
N_TIME_BINS = 32   # Downsample time axis for SHAP (fewer features = faster)
# ─────────────────────────────────────────────────────────────

def compute_logmel(audio, sr=None, n_mels=N_MELS, n_time=N_TIME_BINS):
    """Compute a fixed-size log-Mel spectrogram."""
    if sr is None:
        sr = sampling_rate
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel, ref=np.max)  # [n_mels, T]
    # Resize time axis to fixed n_time bins via simple averaging
    T = log_mel.shape[1]
    step = max(T // n_time, 1)
    log_mel = np.stack([
        log_mel[:, i*step:(i+1)*step].mean(axis=1)
        for i in range(n_time)
    ], axis=1)  # [n_mels, n_time]
    return log_mel

def logmel_to_audio_approx(log_mel_flat, original_audio, n_mels=N_MELS, n_time=N_TIME_BINS):
    """
    Approximate reconstruction: scale original audio by per-segment energy mask.
    This is an approximation — true Griffin-Lim inversion is expensive.
    """
    log_mel = log_mel_flat.reshape(n_mels, n_time)  # [n_mels, n_time]
    # Compute per-time-bin energy from the masked spectrogram
    energy_mask = log_mel.mean(axis=0)              # [n_time]
    energy_mask = (energy_mask - energy_mask.min()) / (energy_mask.ptp() + 1e-8)
    # Tile mask to original audio length
    segment_len = len(original_audio) // n_time
    mask = np.repeat(energy_mask, segment_len)[:len(original_audio)]
    return (original_audio * mask).astype(np.float32)

def score_from_spectrogram(spec_flat_batch):
    """SHAP calls this with a batch of flattened spectrograms."""
    audios = [
        logmel_to_audio_approx(row, audio_array)
        for row in spec_flat_batch
    ]
    return predict_proba(audios)  # [batch, num_labels]

# Compute the spectrogram for our audio
log_mel   = compute_logmel(audio_array)              # [N_MELS, N_TIME_BINS]
spec_flat = log_mel.flatten().reshape(1, -1)         # [1, N_MELS * N_TIME_BINS]

print(f"Spectrogram shape : {log_mel.shape}")
print(f"SHAP input dim    : {spec_flat.shape[1]} features")
print("Running SHAP KernelExplainer (this may take a minute)...")

explainer_shap = shap.KernelExplainer(score_from_spectrogram, spec_flat)
shap_values    = explainer_shap.shap_values(spec_flat, nsamples=100)  # [num_labels, 1, features]

# Visualise SHAP map for the top label
shap_map = shap_values[top_label_idx][0].reshape(N_MELS, N_TIME_BINS)  # [n_mels, n_time]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Log-Mel spectrogram
im0 = axes[0].imshow(log_mel, aspect="auto", origin="lower",
                      cmap="magma", interpolation="nearest")
axes[0].set_title("Log-Mel spectrogram")
axes[0].set_xlabel("Time bin")
axes[0].set_ylabel("Mel frequency bin")
plt.colorbar(im0, ax=axes[0], label="dB")

# SHAP attribution map
vmax = np.abs(shap_map).max()
im1  = axes[1].imshow(shap_map, aspect="auto", origin="lower",
                       cmap="RdYlGn", vmin=-vmax, vmax=vmax,
                       interpolation="nearest")
axes[1].set_title(f"SHAP attribution — label: '{CANDIDATE_LABELS[top_label_idx]}'")
axes[1].set_xlabel("Time bin")
axes[1].set_ylabel("Mel frequency bin")
plt.colorbar(im1, ax=axes[1], label="SHAP value")

plt.tight_layout()
plt.show()
print("Green = supports label | Red = opposes label")

## 5. Approach 3 — Captum Integrated Gradients

**What Integrated Gradients does:** It integrates the model's gradients along a straight path from a baseline input (silence) to the actual input. This gives a provably faithful attribution that satisfies the *completeness axiom* — the attributions sum exactly to the difference in output between baseline and input.

**Advantages over SHAP/LIME:**
- No sampling approximation — exact gradient-based
- Much faster (no thousands of forward passes)
- Directly interrogates the model's internal computation

**How we apply it to CLAP:** We hook into the model just after the spectrogram feature extraction, treat the similarity score with a text label as the output, and attribute back to the input spectrogram.

In [ ]:
from captum.attr import IntegratedGradients

# ── Build the spectrogram tensor that CLAP actually uses ─────
inputs_pt = processor(
    audios=audio_array, return_tensors="pt", sampling_rate=sampling_rate
)

# The processor returns input_features: the log-Mel spectrogram [1, 1, n_mels, T]
spec_tensor = inputs_pt["input_features"].clone().requires_grad_(True)
print(f"Spectrogram tensor shape: {spec_tensor.shape}")

# Precompute text embeddings for all labels
text_inputs_pt = processor(text=CANDIDATE_LABELS, return_tensors="pt", padding=True)
with torch.no_grad():
    text_emb_pt = model.get_text_features(**text_inputs_pt)          # [num_labels, 512]
    text_emb_pt = text_emb_pt / text_emb_pt.norm(dim=-1, keepdim=True)

def forward_similarity(spec_input, label_idx=None):
    """
    Forward pass: spectrogram → audio embedding → cosine similarity with one label.
    This is the scalar that Integrated Gradients will attribute.
    """
    if label_idx is None:
        label_idx = top_label_idx

    # Run only the audio encoder (SWINTransformer + projection)
    audio_out  = model.audio_model(spec_input)          # uses input_features internally
    audio_emb  = model.audio_projection(audio_out.pooler_output)  # [B, 512]
    audio_emb  = audio_emb / audio_emb.norm(dim=-1, keepdim=True)

    # Similarity with the target label
    sim = (audio_emb @ text_emb_pt[label_idx].unsqueeze(-1)).squeeze(-1)  # [B]
    return sim

# Baseline: silence (all zeros)
baseline = torch.zeros_like(spec_tensor)

# Run Integrated Gradients
ig = IntegratedGradients(forward_similarity)
attributions, delta = ig.attribute(
    spec_tensor,
    baselines=baseline,
    n_steps=50,                     # more steps = more accurate, slower
    return_convergence_delta=True,
)

print(f"Convergence delta (should be small): {delta.item():.6f}")

# Collapse to [n_mels, T] for visualisation
attr_map = attributions.squeeze().detach().numpy()  # [n_mels, T] or [1, n_mels, T]
if attr_map.ndim == 3:
    attr_map = attr_map[0]

spec_np = spec_tensor.squeeze().detach().numpy()
if spec_np.ndim == 3:
    spec_np = spec_np[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Spectrogram
im0 = axes[0].imshow(spec_np, aspect="auto", origin="lower",
                      cmap="magma", interpolation="nearest")
axes[0].set_title("CLAP input spectrogram")
axes[0].set_xlabel("Time frame")
axes[0].set_ylabel("Mel frequency bin")
plt.colorbar(im0, ax=axes[0])

# Integrated Gradients attribution
vmax = np.abs(attr_map).max()
im1  = axes[1].imshow(attr_map, aspect="auto", origin="lower",
                       cmap="RdYlGn", vmin=-vmax, vmax=vmax,
                       interpolation="nearest")
axes[1].set_title(f"Integrated Gradients — label: '{CANDIDATE_LABELS[top_label_idx]}'")
axes[1].set_xlabel("Time frame")
axes[1].set_ylabel("Mel frequency bin")
plt.colorbar(im1, ax=axes[1], label="Attribution")

plt.tight_layout()
plt.show()

## 6. Side-by-side comparison

Here we overlay all three attribution methods on the same spectrogram to compare how they agree or disagree on which time-frequency regions matter most.

In [ ]:
import matplotlib.gridspec as gridspec
from scipy.ndimage import zoom

def resize_map(arr, target_shape):
    """Resize a 2D attribution map to target_shape using zoom."""
    factors = (target_shape[0] / arr.shape[0], target_shape[1] / arr.shape[1])
    return zoom(arr, factors)

# Reference shape: the Captum spectrogram (most faithful to model input)
ref_shape = attr_map.shape  # [n_mels, T]

# Resize LIME weights to spectrogram shape (LIME has only N_SEGMENTS in time)
lime_2d = np.tile(weight_array, (N_MELS, 1))  # [N_MELS, N_SEGMENTS]
lime_resized = resize_map(lime_2d, ref_shape)

# Resize SHAP map
shap_resized = resize_map(shap_map, ref_shape)

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.35)

methods = [
    (spec_np,       "Input spectrogram",                    "magma",  None,    None),
    (lime_resized,  "LIME\n(time segments)",                "RdYlGn", None,    None),
    (shap_resized,  "SHAP\n(spectrogram regions)",          "RdYlGn", None,    None),
    (attr_map,      "Captum\n(Integrated Gradients)",       "RdYlGn", None,    None),
]

for i, (data, title, cmap, vmin, vmax) in enumerate(methods):
    ax = fig.add_subplot(gs[i])
    if cmap == "RdYlGn":
        v = np.abs(data).max()
        vmin, vmax = -v, v
    im = ax.imshow(data, aspect="auto", origin="lower",
                   cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Time frame")
    if i == 0:
        ax.set_ylabel("Mel frequency bin")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(
    f"Attribution comparison — label: '{CANDIDATE_LABELS[top_label_idx]}'",
    fontsize=13, y=1.02
)
plt.show()

## 7. Tips and next steps

### Improving accuracy
- **LIME:** Increase `num_samples` (500–1000) for more stable explanations
- **SHAP:** Increase `nsamples` (200–500) — but expect longer runtimes
- **Captum:** Increase `n_steps` (100–200) for more accurate integration

### Better audio perturbation
The spectrogram→audio reconstruction used here is an approximation. For higher fidelity, replace `logmel_to_audio_approx` with Griffin-Lim inversion:
```python
import librosa
audio_reconstructed = librosa.feature.inverse.mel_to_audio(mel_spectrogram, sr=sampling_rate)
```

### Using your own audio
Replace the LibriSpeech sample with any audio file:
```python
import librosa
audio_array, sampling_rate = librosa.load("your_file.wav", sr=48000, mono=True)
audio_array = audio_array.astype(np.float32)
```

### Choosing the right method
| Question | Best approach |
|---|---|
| Which part of the recording drives the prediction? | LIME (time segments) |
| Which frequency bands matter at which times? | SHAP or Captum (spectrogram) |
| Need speed and faithfulness to model internals? | Captum IntegratedGradients |
| No PyTorch / framework-agnostic setup? | SHAP KernelExplainer |